In [0]:
%sql
create schema if not exists apexlife.silver;
     

In [0]:
%sql
select * from apexlife.bronze.diagnosis_raw limit 5

diagnosis_code,diagnosis_desc
D001,Hypertension
D002,Diabetes Type 2
D003,Chest Pain
D004,Asthma
D005,Kidney Infection


In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_table = 'apexlife.bronze.diagnosis_raw'
silver_table = 'apexlife.silver.dim_diagnosis'
checkpoint_path = "abfss://data@apexlife.dfs.core.windows.net/silver/dim_diagnosis/checkpoint/"

In [0]:
df = (
    spark.readStream.table(bronze_table)
)

df = (
    df
    .dropDuplicates(["diagnosis_code"])
    .withColumn('load_time' , current_timestamp() )
)

In [0]:
# This imports the Delta Lake API that allows you MERGE , UPDATE, DELETE, and INSERT 
from delta.tables import DeltaTable

In [0]:
def merge_dim_diagnosis(batch_df, batch_id):

    if spark.catalog.tableExists(silver_table):

        dim_diagnosis = DeltaTable.forName(spark, silver_table)

        (
            dim_diagnosis.alias("t")
            .merge(
                batch_df.alias("s"),
                "t.diagnosis_code = s.diagnosis_code"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:

        batch_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(silver_table)


(
    df.writeStream
        .foreachBatch(merge_dim_diagnosis)
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql
select * from apexlife.silver.dim_diagnosis

diagnosis_code,diagnosis_desc,load_time
D005,Kidney Infection,2026-06-03T18:40:53.133Z
D002,Diabetes Type 2,2026-06-03T18:40:53.133Z
D001,Hypertension,2026-06-03T18:40:53.133Z
D003,Chest Pain,2026-06-03T18:40:53.133Z
D004,Asthma,2026-06-03T18:40:53.133Z
